In [3]:
#数据处理模块，以及依赖库导入
%run data_process.ipynb
from feature_extractor.features_extractor import Data_features_extractor

import tensorflow as tf
import time
from net.base_cnn import *
from net.triplet import triplet_net,triplet_loss
from tensorflow.keras import optimizers


from sklearn import datasets       #导入数据模块
from sklearn.model_selection import train_test_split     #导入切分训练集、测试集模块
from sklearn.neighbors import KNeighborsClassifier

from sklearn.manifold import TSNE
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt
import numpy as np
import joblib

In [4]:
#按照顺序，不打乱的提取去一定格式的训练和验证数据
#获取三类训练数据
#获取三类验证数据

# BASE_MODE = "mfcc-base"
# DELTA_MODE = "mfcc-delta"
# DELTA_DELTA_MODE = "mfcc-delta-delta"
# RASTA_MODE = "raste"
# PLP_MODE = "plp"
# RASTA_PLP_MODE = "raste-plp"
# MFCC_RASTA_MODE = "mfcc-raste"

#加载深度学习模型
Bc_one=tf.keras.models.load_model('save_model/model_one/35')
Bc_two=tf.keras.models.load_model('save_model/model_two/99')

#加载机器学习模型
xgb_one = joblib.load('step_one.model')
xgb_two = joblib.load('step_two.model')

#加载数据
Data_for_one=get_datas_for_test_NOW("plp",'test',Bc_one,All=False,method='audio')
Data_for_two=get_datas_for_test_NOW("plp",'test',Bc_two,All=False,method='feature')

D:\anaconda\lib\site-packages\xgboost\compat.py:31: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


In [5]:
Data_for_one[0][0][0].shape

(32,)

In [15]:
ad=0
hc=0
n_ad=0
mci=0
#三类音频
for p in [0,1,2]:
    for m in range(len(Data_for_one[p])):#j指类中的数组对
        Audio=Data_for_one[p][m]

        pred=xgb_one.predict(Audio)#pred为预测的标签
        a=0
        b=0        
        #计算预测的的结果
        #0为ad 1为n_ad
        for k in pred:
            if(k==0):
                a+=1
            if(k!=0):
                b+=1
        
        res=b/(a+b)   
        if(res<=0.36 and p==0):#认定为识别正确的ad
            ad+=1
        #print(ad)
        if(res>0.36 and p!=0):#认定为识别正确的n_ad,进而进行第二次分类
            n_ad+=1
            Audio=Data_for_two[p][m]
            pred=xgb_two.predict(Audio)
            a=0
            b=0        
            #计算预测的的结果
            #0为hc 1为mci
            for k in pred:
                if(k==0):
                    a+=1
                if(k!=0):
                    b+=1
            res=b/(a+b)   
            if(res<=0.4 and p==1):#认定为hc
                hc+=1
            if(res>0.4 and p!=1):#否则认定为mci,进而进行第二次分类
                mci+=1   

acc_ad=ad/np.array(Data_for_two[0]).shape[0]
acc_n_ad=n_ad/(np.array(Data_for_two[1]).shape[0]+np.array(Data_for_two[2]).shape[0])

acc_hc=hc/np.array(Data_for_two[1]).shape[0]
acc_mci=mci/np.array(Data_for_two[2]).shape[0]

#ad准确率
print("ad:",ad/np.array(Data_for_two[0]).shape[0])

#n_ad准确率
print("n_ad:",acc_n_ad)

#hc准确率
print("hc:",hc/np.array(Data_for_two[1]).shape[0])

#mci准确率
print("mci:",mci/np.array(Data_for_two[2]).shape[0])

print("average:",(acc_ad+acc_hc+acc_mci)/3)

ad: 0.8857142857142857
n_ad: 0.8809523809523809
hc: 0.8222222222222222
mci: 0.8205128205128205
average: 0.8428164428164427


C:\Users\dell\AppData\Local\Temp\ipykernel_17992\1660175416.py:44: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  acc_ad=ad/np.array(Data_for_two[0]).shape[0]
C:\Users\dell\AppData\Local\Temp\ipykernel_17992\1660175416.py:45: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  acc_n_ad=n_ad/(np.array(Data_for_two[1]).shape[0]+np.array(Data_for_two[2]).shape[0])
C:\Users\dell\AppData\Local\Temp\ipykernel_17992\1660175416.py:47: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays w

In [11]:
hc=0
mci=0
#三类音频
for p in [1,2]:
    for m in range(len(Data_for_two[p])):#j指类中的数组对
        Audio=Data_for_two[p][m]

        pred=xgb_two.predict(Audio)#pred为预测的标签
        a=0
        b=0        
        #计算预测的的结果
        #0为hc 1为mci
        for k in pred:
            if(k==0):
                a+=1
            if(k==1):
                b+=1

        res=b/(a+b)   
        if(res<=0.4 and p==1):#认定为hc
            hc+=1
        if(res>0.4 and p!=1):#否则认定为mci,进而进行第二次分类
            mci+=1

#hc准确率
print("hc:",hc/np.array(Data_for_two[1]).shape[0])

#mci准确率
print("mci:",mci/np.array(Data_for_two[2]).shape[0])

print("average:",(hc/np.array(Data_for_two[1]).shape[0]+mci/(np.array(Data_for_two[2]).shape[0]))/2)

hc: 0.8888888888888888
mci: 0.9487179487179487
average: 0.9188034188034188


C:\Users\dell\AppData\Local\Temp\ipykernel_16524\2380847488.py:26: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  print("hc:",hc/np.array(Data_for_two[1]).shape[0])
C:\Users\dell\AppData\Local\Temp\ipykernel_16524\2380847488.py:29: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  print("mci:",mci/np.array(Data_for_two[2]).shape[0])
C:\Users\dell\AppData\Local\Temp\ipykernel_16524\2380847488.py:31: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or sh

In [4]:
xgb_one = joblib.load('step_one.model')

In [14]:
ad=0
n_ad=0
#三类音频
for p in range(3):
    for m in range(len(Data_for_one[p])):#j指类中的数组对
        Audio=Data_for_one[p][m]

        pred=xgb_one.predict(Audio)#pred为预测的标签
        #print(p,m)
        a=0
        b=0        
        #计算预测的的结果
        #0为ad 1为hc_mci
        for k in pred:
            if(k==0):
                a+=1
            if(k==1):
                b+=1

        res=b/(a+b)   
        if(res<=0.36 and p==0):#认定为AD
            ad+=1
        if(res>0.36 and p!=0):#否则认定为HC_MCI,进而进行第二次分类
            n_ad+=1

#ad准确率
print("ad:",ad/np.array(Data_for_one[0]).shape[0])

#n_ad准确率
print("n_ad:",n_ad/(np.array(Data_for_one[1]).shape[0]+np.array(Data_for_one[2]).shape[0]))

print("average:",(ad/np.array(Data_for_one[0]).shape[0]+n_ad/(np.array(Data_for_one[1]).shape[0]+np.array(Data_for_one[2]).shape[0]))/2)

ad: 0.8857142857142857
n_ad: 0.8809523809523809
average: 0.8833333333333333


C:\Users\dell\AppData\Local\Temp\ipykernel_26300\505862350.py:27: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  print("ad:",ad/np.array(Data_for_one[0]).shape[0])
C:\Users\dell\AppData\Local\Temp\ipykernel_26300\505862350.py:30: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  print("n_ad:",n_ad/(np.array(Data_for_one[1]).shape[0]+np.array(Data_for_one[2]).shape[0]))
C:\Users\dell\AppData\Local\Temp\ipykernel_26300\505862350.py:32: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or n